# Single-Agent Pipeline

## 1. Setup

In [1]:
%pip install -q langgraph jsonschema


In [2]:
import random
import re
import time
import json
from typing import TypedDict, List, Optional

from jsonschema import validate, ValidationError


## 2. Graph backend

In [3]:
try:
    from langgraph.graph import StateGraph, END
    BACKEND = "langgraph"
except ImportError:
    BACKEND = "custom"

    END = "__END__"

    class StateGraph:
        """Minimal stand-in for langgraph.graph.StateGraph."""

        def __init__(self, state_schema):
            self.nodes = {}
            self.edges = {}          # name -> next_name (unconditional)
            self.cond_edges = {}     # name -> (router_fn, {label: next_name})
            self.entry = None

        def add_node(self, name, fn):
            self.nodes[name] = fn

        def set_entry_point(self, name):
            self.entry = name

        def add_edge(self, src, dst):
            self.edges[src] = dst

        def add_conditional_edges(self, src, router_fn, mapping):
            self.cond_edges[src] = (router_fn, mapping)

        def compile(self):
            graph = self

            class Runner:
                def invoke(self, state, config=None):
                    current = graph.entry
                    steps = 0
                    while current != END and steps < 50:
                        state = graph.nodes[current](state)
                        if current in graph.cond_edges:
                            router_fn, mapping = graph.cond_edges[current]
                            label = router_fn(state)
                            current = mapping[label]
                        else:
                            current = graph.edges.get(current, END)
                        steps += 1
                    return state

            return Runner()

print("Backend in use:", BACKEND)


Backend in use: langgraph


## 3. State & schemas

In [4]:
class AgentState(TypedDict):
    query: str
    intent: Optional[str]
    tool_output: Optional[dict]
    response: Optional[str]
    trajectory: List[dict]
    retry_count: int
    error: Optional[str]


TOOL_INPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "tool": {"type": "string"},
        "query": {"type": "string"},
    },
    "required": ["tool", "query"],
}

TOOL_OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "tool": {"type": "string"},
        "status": {"type": "string", "enum": ["success", "error"]},
        "result": {},
    },
    "required": ["tool", "status", "result"],
}

MAX_RETRIES = 3


def log_step(state: AgentState, node: str, detail: str) -> None:
    state["trajectory"].append({"node": node, "detail": detail, "t": time.time()})


def _validate(payload, schema, label):
    try:
        validate(instance=payload, schema=schema)
        return True, None
    except ValidationError as e:
        return False, f"{label} schema error: {e.message}"


## 4. Nodes


In [5]:
def analyze_query(state: AgentState) -> AgentState:
    q = state["query"].lower()
    if "calculate" in q or re.search(r"[\d]+\s*[\+\-\*/]\s*[\d]+", q):
        intent = "calculator"
    elif "keyword" in q:
        intent = "keywords"
    else:
        intent = "general"
    state["intent"] = intent
    log_step(state, "analyze_query", f"classified intent as '{intent}'")
    return state


def calculator_tool(state: AgentState) -> AgentState:
    tool_input = {"tool": "calculator", "query": state["query"]}
    ok, err = _validate(tool_input, TOOL_INPUT_SCHEMA, "input")
    if not ok:
        state["error"] = err
        log_step(state, "calculator_tool", err)
        return state

    # Simulate a flaky external call: this is what makes the retry
    # loop (cycle) in the graph actually necessary.
    if random.random() < 0.35:
        state["error"] = "transient calculator API failure"
        state["retry_count"] += 1
        log_step(state, "calculator_tool",
                  f"transient failure (attempt {state['retry_count']})")
        return state

    expr = re.findall(r"[\d\.\+\-\*/\s]+", state["query"])
    try:
        value = eval(expr[0], {"__builtins__": {}}) if expr else None
    except Exception:
        value = None

    output = {"tool": "calculator", "status": "success", "result": value}
    ok, err = _validate(output, TOOL_OUTPUT_SCHEMA, "output")
    if not ok:
        state["error"] = err
        log_step(state, "calculator_tool", err)
        return state

    state["tool_output"] = output
    state["error"] = None
    log_step(state, "calculator_tool", f"computed result = {value}")
    return state


def keyword_tool(state: AgentState) -> AgentState:
    tool_input = {"tool": "keyword_extraction", "query": state["query"]}
    ok, err = _validate(tool_input, TOOL_INPUT_SCHEMA, "input")
    if not ok:
        state["error"] = err
        log_step(state, "keyword_tool", err)
        return state

    stopwords = {"the", "a", "an", "of", "in", "extract", "keywords", "from", "and", "to"}
    words = re.findall(r"[a-zA-Z]+", state["query"].lower())
    keywords = [w for w in words if w not in stopwords]

    output = {"tool": "keyword_extraction", "status": "success", "result": keywords}
    state["tool_output"] = output
    state["error"] = None
    log_step(state, "keyword_tool", f"extracted keywords: {keywords}")
    return state


def general_response(state: AgentState) -> AgentState:
    state["tool_output"] = {"tool": "general", "status": "success",
                             "result": "handled as a general query"}
    log_step(state, "general_response", "no specialised tool needed")
    return state


def error_handler(state: AgentState) -> AgentState:
    log_step(state, "error_handler",
              f"giving up after {state['retry_count']} retries: {state['error']}")
    state["tool_output"] = {"tool": "calculator", "status": "error", "result": None}
    return state


def compose_response(state: AgentState) -> AgentState:
    out = state["tool_output"]
    if out and out.get("status") == "success":
        state["response"] = f"[{out['tool']}] -> {out['result']}"
    else:
        state["response"] = "Sorry, I couldn't complete that request."
    log_step(state, "compose_response", state["response"])
    return state


## 5. Routers (conditional edges)


In [6]:
def route_intent(state: AgentState) -> str:
    return state["intent"]


def route_calculator_retry(state: AgentState) -> str:
    if state["error"] is None:
        return "done"
    if state["retry_count"] < MAX_RETRIES:
        return "retry"
    return "give_up"


## 6. Build the graph

In [7]:
def build_pipeline():
    graph = StateGraph(AgentState)

    graph.add_node("analyze_query", analyze_query)
    graph.add_node("calculator_tool", calculator_tool)
    graph.add_node("keyword_tool", keyword_tool)
    graph.add_node("general_response", general_response)
    graph.add_node("error_handler", error_handler)
    graph.add_node("compose_response", compose_response)

    graph.set_entry_point("analyze_query")

    graph.add_conditional_edges(
        "analyze_query",
        route_intent,
        {
            "calculator": "calculator_tool",
            "keywords": "keyword_tool",
            "general": "general_response",
        },
    )

    # Cycle: calculator_tool -> itself (retry) -> error_handler (give up) -> compose_response
    graph.add_conditional_edges(
        "calculator_tool",
        route_calculator_retry,
        {
            "retry": "calculator_tool",
            "give_up": "error_handler",
            "done": "compose_response",
        },
    )

    graph.add_edge("keyword_tool", "compose_response")
    graph.add_edge("general_response", "compose_response")
    graph.add_edge("error_handler", "compose_response")
    graph.add_edge("compose_response", END)

    return graph.compile()


def run_query(app, query: str) -> AgentState:
    initial: AgentState = {
        "query": query,
        "intent": None,
        "tool_output": None,
        "response": None,
        "trajectory": [],
        "retry_count": 0,
        "error": None,
    }
    return app.invoke(initial)


## 7. Try it out

In [8]:
random.seed(7)
app = build_pipeline()

test_queries = [
    "calculate 12 + 30",
    "extract keywords from the quick brown fox jumps",
    "what is the capital of France",
    "calculate 100 / 4",
]

for q in test_queries:
    result = run_query(app, q)
    print(f"\nQuery: {q}")
    print("Response:", result["response"])
    print("Trajectory:")
    for step in result["trajectory"]:
        print("  -", step["node"], ":", step["detail"])



Query: calculate 12 + 30
Response: [calculator] -> 42
Trajectory:
  - analyze_query : classified intent as 'calculator'
  - calculator_tool : computed result = 42
  - compose_response : [calculator] -> 42

Query: extract keywords from the quick brown fox jumps
Response: [keyword_extraction] -> ['quick', 'brown', 'fox', 'jumps']
Trajectory:
  - analyze_query : classified intent as 'keywords'
  - keyword_tool : extracted keywords: ['quick', 'brown', 'fox', 'jumps']
  - compose_response : [keyword_extraction] -> ['quick', 'brown', 'fox', 'jumps']

Query: what is the capital of France
Response: [general] -> handled as a general query
Trajectory:
  - analyze_query : classified intent as 'general'
  - general_response : no specialised tool needed
  - compose_response : [general] -> handled as a general query

Query: calculate 100 / 4
Response: [calculator] -> 25.0
Trajectory:
  - analyze_query : classified intent as 'calculator'
  - calculator_tool : transient failure (attempt 1)
  - calcul

## 8. Evaluation

In [9]:
def evaluate(app, queries: List[str]) -> dict:
    completed = 0
    total_tool_calls = 0
    total_latency = 0.0
    results = []

    for q in queries:
        start = time.time()
        result = run_query(app, q)
        elapsed = time.time() - start
        total_latency += elapsed

        tool_calls = sum(1 for s in result["trajectory"]
                          if s["node"] in ("calculator_tool", "keyword_tool", "general_response"))
        total_tool_calls += tool_calls

        success = result["tool_output"] is not None and result["tool_output"]["status"] == "success"
        completed += int(success)

        results.append({
            "query": q,
            "success": success,
            "tool_calls": tool_calls,
            "latency_s": round(elapsed, 4),
        })

    n = len(queries)
    return {
        "task_completion_rate": completed / n if n else 0.0,
        "avg_tool_calls_per_query": total_tool_calls / n if n else 0.0,
        "avg_latency_s": total_latency / n if n else 0.0,
        "per_query": results,
    }


metrics = evaluate(app, test_queries)
print(json.dumps(metrics, indent=2))


{
  "task_completion_rate": 1.0,
  "avg_tool_calls_per_query": 1.25,
  "avg_latency_s": 0.008220076560974121,
  "per_query": [
    {
      "query": "calculate 12 + 30",
      "success": true,
      "tool_calls": 2,
      "latency_s": 0.0138
    },
    {
      "query": "extract keywords from the quick brown fox jumps",
      "success": true,
      "tool_calls": 1,
      "latency_s": 0.0064
    },
    {
      "query": "what is the capital of France",
      "success": true,
      "tool_calls": 1,
      "latency_s": 0.0033
    },
    {
      "query": "calculate 100 / 4",
      "success": true,
      "tool_calls": 1,
      "latency_s": 0.0094
    }
  ]
}


## 9. Notes: how this maps back to the quiz

| Quiz question | Where it shows up here |
|---|---|
| Q1 stateful directed graph vs linear pipeline | `AgentState` is threaded through every node; routing can branch and loop |
| Q2 nodes & edges | `add_node` / `add_edge` calls in `build_pipeline` |
| Q3 conditional routing | `route_intent` + `add_conditional_edges` on `analyze_query` |
| Q4 retry loop / cycle | `route_calculator_retry`: `calculator_tool` can route back to itself |
| Q5 single agent simulating multiple roles | `analyze_query` / tool nodes / `compose_response` act like separate specialised agents inside one system |
| Q6 JSON schema tools | `TOOL_INPUT_SCHEMA`, `TOOL_OUTPUT_SCHEMA`, validated with `jsonschema` |
| Q7 sequential vs parallel calls | This pipeline is sequential (each node depends on the previous state); noted here as a discussion point since all tool calls here are independent per query and *could* be parallelised across queries |
| Q8 error handling | try/except-style schema validation + `error_handler` node + retry mechanism |
| Q9 trajectory evaluation | `trajectory` log inspected in section 7 |
| Q10 task completion rate & cost metrics | `evaluate()` in section 8 |
